<a href="https://colab.research.google.com/github/RyantRamadhan/BigData26_B_2411532001_Imam-Aryanta-Ramadhan/blob/main/Praktikum02/BD_B_P02_2411532001_Imam_Aryanta_Ramadhan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 9.3 MB/s eta 0:00:00


1. Import Library dan Inisialisasi

In [55]:
import numpy as np
import pandas as pd
from faker import Faker
import random

In [56]:
from google.colab import drive
drive.mount("/content/drive")

import os
DIR_KERJA  = "/content/data"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['transaksi_mentah.csv', 'transaksi_bersih.csv']


2. Membuat Dataset Sintetis (Simulasi Data Acquisition)

In [41]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f"{harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional
    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })
df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
        idx = df.sample(frac=frac, random_state=SEED).index

        df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))


Jumlah baris: 515


In [42]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating
0,TRX00305,Ophelia Hartati,Quo Basic,Buku,50000,4,kartu kredit,2026-07-15,Blitar,1.0
1,TRX00500,Cut Maya Wijayanti,Delectus,Elektronik,Rp500.000,1,E-Wallet,2026-07-11,Lubuklinggau,NaN
2,TRX00442,"Dr. Ridwan Utama, M.Pd",Animi Max,Elektronik,25000,1,COD,2026-08-27,Probolinggo,2.0
3,TRX00154,"Viman Suwarno, S.H.",Consectetur,Fashion,Rp250.000,1,COD,29/07/2026,Tual,4.0
4,TRX00074,NaN,Eius,Olahraga,250000.0,4,NaN,03/07/2026,NaN,1.0


3. Deteksi dan Penanganan Missing Value

In [43]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [44]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah drop_duplicates():", len(df))

Jumlah baris setelah drop_duplicates(): 495


4. Deteksi dan Penanganan Duplicate

In [45]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


5. Koreksi Tipe Data dan Standardisasi Format

In [46]:
# Standardisasi teks kategorikal
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [47]:
# Koreksi tipe data pada kolom price
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

In [48]:
# Standardisasi format tanggal ke YYYY-MM-DD:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [49]:
# Finalisasi tipe data
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

6. Ekspor Dataset Bersih

In [51]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


In [52]:
df['is_valid_price'] = df['price'] > 0

invalid_prices = df[df['is_valid_price'] == False]
print(f"Jumlah transaksi dengan harga tidak valid: {len(invalid_prices)}")

Jumlah transaksi dengan harga tidak valid: 0


In [53]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating,is_valid_price
0,TRX00305,Ophelia Hartati,Quo Basic,Buku,50000.0,4,Kartu Kredit,2026-07-15,Blitar,1.0,True
1,TRX00500,Cut Maya Wijayanti,Delectus,Elektronik,500000.0,1,E-Wallet,2026-07-11,Lubuklinggau,NaN,True
2,TRX00442,"Dr. Ridwan Utama, M.Pd",Animi Max,Elektronik,25000.0,1,COD,2026-08-27,Probolinggo,2.0,True
3,TRX00154,"Viman Suwarno, S.H.",Consectetur,Fashion,250000.0,1,COD,2026-07-29,Tual,4.0,True
5,TRX00132,Ir. Ilyas Setiawan,Ducimus Pro,Rumah Tangga,250000.0,4,COD,2026-09-17,Subulussalam,2.0,True


In [54]:
transaksi_per_kategori = df['category'].value_counts()
print(transaksi_per_kategori)

category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64
